In [1]:
%pip install xgboost
# Manipulación de datos y tiempo
import pandas as pd
import numpy as np 
import sys
import os

# Agregamos la carpeta 'src' a las rutas de búsqueda de Python
sys.path.append(os.path.abspath("../src"))
# Divisiónes temporales, validación cruzada
from sklearn.model_selection import train_test_split, TimeSeriesSplit
# Modelos 
from sklearn.linear_model import LinearRegression, Ridge, Lasso 
# Modelo forest
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb 
# Metricas de exito o negatividad
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
#modelos tontos xd
from sklearn.dummy import DummyRegressor
#para recargar si el .py sufre variaciones
%load_ext autoreload
%autoreload
import funciones
from funciones import evaluar_modelo


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# 1. División Temporal (Sin shuffle para no alterar la línea de tiempo)
df = pd.read_csv('../data/dataset_rutas_limpios.csv')
X = df.drop(columns=['volumen_total_cm3']) 
y = df['volumen_total_cm3']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, shuffle=False)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

print(
    f"Entrenamiento corresponde a: {X_train.index.min()} hasta {X_train.index.max()}"
)
print(
    f"Validación corresponde a:    {X_test.index.min()} hasta {X_test.index.max()}"
)
print("-" * 50)




Entrenamiento corresponde a: 0 hasta 2815
Validación corresponde a:    2816 hasta 3519
--------------------------------------------------


In [3]:
#Ahora crearemos un modelo burdo, en donde se hara manualmente si quiero saber la demanda de mañana puedo estimarla con el promedio(simple, movil,estacionalidad), 
#por lo cual el mean_squared_error de nuestro modelo debe ser menor que este o si no no vale la pena implementarlo
#Por lo cual haremos un modelo base (baseline) estrategia promedio simple

modelo_base = DummyRegressor(strategy='mean')
modelo_base.fit(X_train, y_train)
predicciones_base = modelo_base.predict(X_test)

# Evaluamos las metricas 
from funciones import evaluar_modelo

In [4]:
# 1. Vuelves a cargar los datos
df = pd.read_csv('../data/dataset_rutas_limpios.csv')

# 2. LA CLAVE: Pasas la fecha al índice para que no sea una variable predictora
# (Cambia 'fecha' por el nombre exacto de la columna de fechas en tu archivo)
df.set_index('fecha', inplace=True) 

# 3. Separas los datos (ahora X ya no tiene fechas, solo números)
X = df.drop(columns=['volumen_total_cm3']) 
y = df['volumen_total_cm3']
X = pd.get_dummies(X, drop_first=True)
# 4. Divides para entrenar y validar
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [5]:
#creacion y entranamiento de arbol
#primera opcion 
rf_model = RandomForestRegressor(
    n_estimators = 100,
    random_state = 42
)
#segunda opcion 
xgb_model = xgb.XGBRegressor(
    n_estimators = 100,
    learning_rate= 0.1,
    random_state=42
)

#entranamiento
rf_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)

# Validacion 
predicciones_rf = rf_model.predict(X_test)
predicciones_xgb = xgb_model.predict(X_test)

#Revisamos las metricas con la funcion 
evaluar_modelo("Random Forest", y_test, predicciones_rf)
evaluar_modelo("XGBoost", y_test, predicciones_xgb)

---Desempeño: Random Forest---
MAPE: 34.31% de error promedio
MAE: 49600.92 paquetes de error promedio
RMSE: 96913.24
------------------------------
---Desempeño: XGBoost---
MAPE: 33.29% de error promedio
MAE: 48745.19 paquetes de error promedio
RMSE: 96367.46
------------------------------
